In [ ]:
%run ./_config

In [ ]:
# The agent, in one place
#
# Labs 2 and 3 load this with `%run ./_agent_core`.
#
# An agent is an LLM + instructions + tools, in a loop:
#
# 1. Send the question, instructions, and tool descriptions to the LLM
# 2. If the LLM asks for a tool, run it (as *you*, so governance applies) and send back the result
# 3. Repeat until the LLM answers in plain English
#
# Docs
# - AI Playground: https://learn.microsoft.com/en-us/azure/databricks/large-language-models/ai-playground
# - MLflow Tracing: https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/overview
#
# Imports and MLflow experiment
#
# Loads the OpenAI-compatible serving client and turns on MLflow autolog so every LLM call appears in the experiment trace.

import json, mlflow
from mlflow.entities import SpanType

mlflow.set_experiment(EXPERIMENT_PATH)
mlflow.openai.autolog()                       # every LLM call shows up in the trace
llm = get_llm_client()                        # Unity Gateway or legacy endpoint (from _config)
print(f"LLM access mode: {LLM_ACCESS_MODE}  model={LLM_ENDPOINT}")


In [ ]:
# Instructions v1 (what most people write first)

INSTRUCTIONS_V1 = """You are an auto-loan servicing assistant for servicing analysts.
Answer questions about loan accounts using the tools provided.
Servicing notes capture the latest conversation with the customer, so use them for the most up-to-date account status.
Be concise and helpful."""

In [ ]:
# The tools: the LLM only sees these names, descriptions, and parameters

TOOLS = [
  {"type": "function", "function": {
    "name": "find_accounts",
    "description": "Find loan and lease accounts in a state within a days-past-due range and above a minimum balance. "
                   "Returns up to 50 rows ordered by balance (highest first) with product, apr, risk_tier, days_past_due, "
                   "bucket, and total_matches (the full count).",
    "parameters": {"type": "object", "properties": {
        "state": {"type": "string", "description": "Two-letter US state code, e.g. TX"},
        "min_dpd": {"type": "integer", "description": "Minimum days past due, inclusive"},
        "max_dpd": {"type": "integer", "description": "Maximum days past due, inclusive"},
        "min_balance": {"type": "number", "description": "Minimum current balance in USD"}},
      "required": ["state", "min_dpd", "max_dpd", "min_balance"]}}},
  {"type": "function", "function": {
    "name": "get_account_360",
    "description": "Full view of one account: balance, APR, risk tier, days past due, bucket, last servicing note, last payment.",
    "parameters": {"type": "object", "properties": {
        "account_id": {"type": "string", "description": "Account identifier, e.g. A100001"}},
      "required": ["account_id"]}}},
  {"type": "function", "function": {
    "name": "payment_history",
    "description": "Payments for one account over the last N months, newest first, with window_total and window_count.",
    "parameters": {"type": "object", "properties": {
        "account_id": {"type": "string", "description": "Account identifier, e.g. A100001"},
        "months": {"type": "integer", "description": "Months back from today"}},
      "required": ["account_id", "months"]}}},
]

# Each tool maps to the Unity Catalog function you registered in Lab 1. Parameters are bound, never pasted into SQL.
TOOL_SQL = {
  "find_accounts":   f"SELECT * FROM {FQ}.find_accounts(:state, :min_dpd, :max_dpd, :min_balance)",
  "get_account_360": f"SELECT * FROM {FQ}.get_account_360(:account_id)",
  "payment_history": f"SELECT * FROM {FQ}.payment_history(:account_id, :months)",
}

In [ ]:
# Running a tool (as you) and the agent loop

def call_tool(name, args):
    """Runs one UC function as the signed-in user. Traced as a RETRIEVER span so MLflow can judge groundedness."""
    with mlflow.start_span(name=f"tool:{name}", span_type=SpanType.RETRIEVER) as span:
        span.set_inputs(args)
        if name not in TOOL_SQL:
            docs = [{"page_content": f"Unknown tool {name}", "metadata": {"doc_uri": f"{name}/error"}}]
        else:
            if "min_balance" in args: args["min_balance"] = float(args["min_balance"])
            for k in ("min_dpd", "max_dpd", "months"):
                if k in args: args[k] = int(args[k])
            rows = [r.asDict() for r in spark.sql(TOOL_SQL[name], args=args).limit(50).collect()]
            docs = [{"page_content": json.dumps(r, default=str),
                     "metadata": {"doc_uri": f"{name}/{r.get('account_id') or r.get('payment_id')}"}} for r in rows] or \
                   [{"page_content": "NO ROWS RETURNED: no accessible records match.", "metadata": {"doc_uri": f"{name}/empty"}}]
        span.set_outputs(docs)
        return docs


@mlflow.trace(span_type=SpanType.AGENT, name="servicing_agent")
def run_agent(question: str, instructions: str = None, verbose: bool = False) -> str:
    messages = [{"role": "system", "content": instructions or INSTRUCTIONS_V1},
                {"role": "user", "content": question}]
    for _ in range(8):
        msg = llm.chat.completions.create(model=LLM_ENDPOINT, messages=messages, tools=TOOLS,
                                          max_tokens=1500, temperature=0).choices[0].message
        if not msg.tool_calls:
            return msg.content
        # Omit assistant content when tool_calls are present. Some FM endpoints
        # (Claude OpenAI-compat) reject content+tool_calls as "assistant prefill".
        turn = {"role": "assistant", "content": None, "tool_calls": [
            {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments or "{}"}}
            for tc in msg.tool_calls]}
        messages.append(turn)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            docs = call_tool(tc.function.name, args)
            if verbose:
                print(f"  🔧 {tc.function.name}({args}) returned {len(docs)} row(s)")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": "\n".join(d["page_content"] for d in docs)})
    return "I stopped after too many tool calls without reaching an answer."

print(f"Agent ready: model={LLM_ENDPOINT}, tools={[t['function']['name'] for t in TOOLS]}")
